# 01 — Cohort and paired-valid target audit

Cohort eligibility is determined by the locked, outcome-independent pose-QC
rules. The signed target is computed on observed bilateral transitions only;
invalid-coordinate sentinels and model-input interpolation do not define it.
Mirroring swaps anatomical sides and must negate both every usable pair
contrast and the aggregate target. Mirrored samples remain paired checks,
never additional independent observations.

In `smoke` profile the cohort is synthetic. Passing these assertions shows
only that the implementation obeys its contract, not empirical performance
on GAVD.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
from collections import Counter

import numpy as np

from laterality.data import prepare_cohort, save_cohort
from laterality.geometry import anatomical_mirror
from laterality.visualization import cohort_figure

cohort = prepare_cohort(context)
reconstructed_targets = np.asarray(
    [
        np.mean(row[np.isfinite(row)])
        for row in cohort.pair_contrasts
    ],
    dtype=np.float64,
)
assert np.allclose(
    reconstructed_targets,
    cohort.table["target"].to_numpy(dtype=np.float64),
    rtol=0.0,
    atol=1e-12,
)

target_contract = cohort.attrition["target_contract"]
assert target_contract["checked_finite_targets"] >= len(cohort.table)
assert target_contract["maximum_mirror_antisymmetry_error"] <= 1e-10
assert target_contract["maximum_invalid_sentinel_error"] <= 1e-12

checked_involutions = 0
for xyz, valid in zip(cohort.model_xyz, cohort.model_valid):
    mirrored_xyz, mirrored_valid = anatomical_mirror(xyz, valid)
    restored_xyz, restored_valid = anatomical_mirror(
        mirrored_xyz, mirrored_valid
    )
    assert np.array_equal(restored_xyz, xyz)
    assert np.array_equal(restored_valid, valid)
    checked_involutions += 1

artifact_paths = save_cohort(context, cohort)
exclusion_reason_counts = Counter(
    item["reason"] for item in cohort.attrition["exclusions"]
)
attrition_summary = {
    key: value
    for key, value in cohort.attrition.items()
    if key != "exclusions"
}
attrition_summary["exclusion_reason_counts"] = dict(
    sorted(exclusion_reason_counts.items())
)

show_inline(cohort_figure(context, cohort))
{
    "attrition": attrition_summary,
    "cohort_digest": cohort.cohort_digest,
    "target_contract": target_contract,
    "checked_model_lane_mirror_involutions": checked_involutions,
    "artifacts": {key: str(value) for key, value in artifact_paths.items()},
}